<a href="https://colab.research.google.com/github/akimotolab/CMAES_Tutorial/blob/main/0_black_box_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 什么是黑盒优化

黑盒优化（Black-Box Optimization, BBO）是指：当用于衡量输入（即解、设计变量）$x$ 优劣的指标（即目标函数值、代价）$f(x)$ 是一个黑盒时，寻找能够优化 $f(x)$ 的 $x$ 的问题。这里所谓目标函数是黑盒，是指给定 $x$ 后只能得到对应的 $f(x)$，通常具有以下特征：
- 无法利用梯度等微分信息（derivative-free optimization；DFO）
- 无法显式地（用数学形式）写出目标函数，因此也无法显式利用目标函数的结构
- 无法利用描述目标函数特性的 Lipschitz 平滑参数等信息（因此也不应依赖这类信息来设置超参数）

下面通过实际求解一个黑盒优化问题来理解 BBO。

## 自己动手求解一个 BBO 问题

先尝试亲手求解一个 BBO 问题。
请先不要查看下面代码的内容，运行后立即将其折叠隐藏。

### 问题定义（运行后隐藏）

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class BBO:
    def __init__(self):
        self._hist = np.empty((0, 3))
    def __call__(self, x, y):
        gx = self.g(x, y)
        fx = self.f(x, y)
        if gx > 0:
            return np.inf
        else:
            self._hist = np.vstack((self._hist, np.array([x, y, fx])))
            return fx
    def plot(self):
        fig = plt.figure()
        plt.scatter(x=self._hist[:, 0], y=self._hist[:, 1], c=self._hist[:, 2], cmap='Blues', edgecolors='blue')
        plt.colorbar()
        fig.gca().set_aspect('equal')
        plt.ylim((0, 1))
        plt.xlim((0, 1))
        plt.xlabel('x')
        plt.ylabel('y')
    def g(self, x, y):
        return max(np.abs(x - 0.5) - 0.5, np.abs(y - 0.5) - 0.5)
    def f(self, x, y):
        a = 2.0
        scale = 3.0
        offset_x = 0.2
        offset_y = - 0.1
        xx = 10 * (x - offset_x) - 5
        yy = 10 * (y - offset_y) - 5
        xxx = (xx + np.sqrt(3) * yy) / 2
        yyy = scale * (np.sqrt(3) * xx - yy) / 2
        return np.sqrt(xxx**2 + a * (1.0 - np.cos(2 * np.pi * xxx)) + yyy**2 + a * (1.0 - np.cos(2 * np.pi * yyy)))
    def contourf(self):
        delta = 0.01
        x = y = np.arange(0.0, 1.0, delta)
        X, Y = np.meshgrid(x, y)
        Z = self.f(X, Y)
        fig = plt.figure()
        plt.contourf(X, Y, Z, 100, cmap='rainbow')
        plt.colorbar()
        fig.gca().set_aspect('equal')
        plt.ylim((0, 1))
        plt.xlim((0, 1))
        plt.xlabel('x')
        plt.ylabel('y')
    def get_hist(self):
        return np.array(self._hist, copy=True)

### 执行脚本
这里考虑的是一个二维优化问题。

`bbo(x, y)`：计算 $f(x, y)$。
每个变量的取值范围均为 $[0, 1]$。
如果输入该范围之外的值，则会被视为不可行，目标函数值返回 $\infty$。

`bbo.plot()`：显示截至目前已经评估过的可行解散点图。

`bbo.get_hist()`：返回截至目前已经评估过的可行解二维数组，每一行为 [$x$, $y$, $f(x, y)$]。


In [ ]:
bbo = BBO()

下面不断改变 $x, y$，尝试猜测最优解。
首先评估定义域中心的位置。

In [ ]:
bbo(0.5, 0.5)
bbo.plot()

再评估几个靠近边界的点。

In [ ]:
bbo(0.1, 0.1)
bbo(0.1, 0.9)
bbo(0.9, 0.1)
bbo(0.9, 0.9)
bbo.plot()
print(bbo.get_hist())

接下来请自己继续猜测并反复评估。
改变 x 和 y，多尝试几次。

In [ ]:
bbo(0.1, 0.1)
bbo.plot()

这个问题的最优解是 $(0.7, 0.4)$，并且 $f(0.7, 0.4) = 0$。
其等高线如下所示。

In [ ]:
bbo.contourf()

在 BBO 中，寻找优质解的线索只有截至目前已经评估过的解及其目标函数值的组合 $\{(x, f(x))\}$。
因此，高效利用这些信息非常重要。

## 尝试穷举搜索

解决这类问题的一种直观思路，是尽可能全面地评估目标函数。
下面实际尝试一下。

这里对每个维度的 $[0, 1]$ 区间（包含边界）划分为 $K$ 个点，并对总计 $K^2$ 个网格点进行评估。

In [ ]:
bbo = BBO()
K = 20
x_array = np.linspace(0, 1, num=K)
y_array = np.linspace(0, 1, num=K)
for x in x_array:
    for y in y_array:
        bbo(x, y)

In [ ]:
bbo.plot()

In [ ]:
min(bbo.get_hist()[:, 2])

除非最优解恰好落在网格点上，否则这种方法无法精确找到最优解，不过通常可以得到一个还不错的解。

但这种方法只有在维度非常低时才现实。
如果维度为 $d$，就需要评估 $K^d$ 个解。
通常一次目标函数评估本身就可能需要运行数值模拟等昂贵计算，因此进行海量评估往往不可行。

此外还可以注意到，上面的做法并没有利用过去已经评估过的解的信息。
一种更高效的做法是：先在粗网格上评估，然后据此缩小下一步的搜索区域，再在该区域附近重复这一过程。这样就不必把大量计算花在明显不太有希望的区域（例如左上角区域）上。

## 使用带 1/5 成功规则的 (1+1)-ES 进行优化

下面来看一种传统的进化策略：(1+1)-ES。
(1+1)-ES 非常简单，但也是一种很有力的方法。理论上已经证明：对于强凸、梯度 Lipschitz 连续的目标函数以及它们的单调变换所构成的一类问题，(1+1)-ES 可以实现线性收敛，即解以指数速度趋近最优解。

(1+1)-ES 不断从正态分布 $N(x, \sigma^2)$ 中生成新的解。每生成并评估一个新解，就利用该信息更新正态分布的均值向量 $x$ 和标准差 $\sigma$，使之后更容易生成较好的解。

下面定义 (1+1)-ES。可以看到它非常简单，核心实现实际上不到 10 行。
`ask` 从正态分布中生成并返回一个新解。
`tell` 利用已经评估的解的信息更新分布参数。

In [ ]:
class ES:
    def __init__(self, init_s, init_x, init_fx):
        self.fx = init_fx
        self.x = np.array(init_x, copy=True)
        self.s = init_s
        self.dim = len(self.x)
        self.alpha = np.exp(1.0 / self.dim)
        self.y = np.empty(self.dim)

    def ask(self):
        self.y = self.x + self.s * np.random.randn(self.dim)
        return self.y

    def tell(self, fx):
        if self.fx < fx:
            self.s /= self.alpha**(0.25)
        else:
            self.s *= self.alpha
            self.x = self.y
            self.fx = fx

下面实际运行一下。
需要首先给定正态分布的初值。这里将均值向量初始化为定义域中心，将标准差设为定义域宽度的一半。

In [ ]:
bbo = BBO()
s = 0.5
x = np.ones(2) * 0.5
fx = bbo(x[0], x[1])
es = ES(s, x, fx)

下面是主循环。
在解被评估到 T 次之前，持续重复搜索。

In [ ]:
T = 100
while len(bbo.get_hist()) < T:
    x = es.ask()
    fx = bbo(x[0], x[1])
    es.tell(fx)

In [ ]:
bbo.plot()
print(min(bbo.get_hist()[:, 2]))

这个问题存在很多局部最优解。
因此，(1+1)-ES 并不一定收敛到全局最优解，但可以看到它会逐渐收敛到某个局部最优解。
由于采用随机搜索，每次运行也可能收敛到不同的局部最优解。
查看生成解的目标函数值历史，可以观察到这一收敛过程。

In [ ]:
import matplotlib.pyplot as plt
plt.plot(bbo.get_hist()[:, 2])

### (1+1)-ES 的工作原理

当新解的目标函数值没有变差时，(1+1)-ES 会用这个新生成的解替换正态分布的均值向量。
因此，均值向量对应的目标函数值会保持单调不增。

标准差会进行自适应，使算法大致能够做到“每 5 次生成一次不劣于当前均值向量的解”。
具体做法是：如果没有改进，就将 $\sigma$ 乘以 $\alpha^{-1/4}$；否则将其乘以 $\alpha$。

可以思考一下：为什么没有改进时要减小 $\sigma$，而出现改进时反而要增大 $\sigma$。

如果 $\sigma$ 过大（为便于理解，可以设想一个非常大的 $\sigma$），从正态分布中生成一个目标函数值优于当前均值向量的解的概率会趋近于 $0$。
因此，如果连续多次迭代都无法改进，就有理由怀疑当前正态分布过宽。
此时应减小 $\sigma$。

反过来，如果 $\sigma$ 过小，并且目标函数在当前均值向量附近具有平滑的等高线，那么均值向量邻域内的等高线可以近似为线性函数的等高线。
由于 $\sigma$ 很小，生成的解会以很高概率落在这个局部区域内。
此时正态分布关于均值向量对称，因此大约有 1/2 的概率生成改进解。
虽然这样仍然可以不断更新解，但每次移动的幅度会非常小。
因此需要增大 $\sigma$。

至少对于具有平滑等高线的目标函数，可以根据上述讨论推断，合理的目标改进概率应该位于 0 到 1/2 之间。
(1+1)-ES 将这一改进概率设为 1/5。
这一取值来自理论分析：当目标函数定义为到最优解的距离时，最优成功概率大约为 1/5。
